# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
pd.set_option('display.max_columns', None)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Display all record sets by @id and name
print("Available Record Sets (@id, name):")
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, list its fields
for rs in dataset.metadata.record_sets:
    print(f"\nRecordSet '@id': {rs['@id']}, name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    if fields:
        for f in fields:
            fid = f.get('@id', '<N/A>')
            name = f.get('name', '<no name>')
            dtype = f.get('dataType', '<no dataType>')
            print(f"    Field @id: {fid}, name: {name}, dataType: {dtype}")
    else:
        print("    No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all available record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Record set IDs:", record_set_ids)

# Extract data from each record set into pandas DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    # Some record sets may not return data (e.g. documentation, etc.)
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            continue
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id}, columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    except Exception as e:
        print(f"Could not load data for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and field for EDA
from pprint import pprint

# --- Find a tabular record set to focus on (the main mass spec data) ---
# For example, let's use the first loaded tabular set.
main_rs_id = None
for rsid in dataframes:
    if len(dataframes[rsid]) > 0:
        main_rs_id = rsid
        break

assert main_rs_id is not None, 'No tabular data found in dataset.'

print(f"Using record set: {main_rs_id}")

main_df = dataframes[main_rs_id]
print(main_df.dtypes)

# --- Identify a numeric field to analyze, e.g. a protein abundance or MW ---
numeric_candidates = main_df.select_dtypes('number').columns.tolist()
print('Numeric fields in the selected record set:', numeric_candidates)
if not numeric_candidates:
    print('No numeric fields found. Please check the DataFrame for other analysis.')

# Example, try 'mw' or another likely column
numeric_field = None
for col in main_df.columns:
    if col.lower() in ['mw', 'molecular_weight', 'abundance', 'normalized_abundance', 'coverage', 'peptide_count']:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
if not numeric_field and numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Selected numeric field: {numeric_field}")
elif numeric_field:
    print(f"Selected numeric field: {numeric_field}")

# Filter and normalize
if numeric_field:
    threshold = main_df[numeric_field].mean() # Example: filter above mean
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a key field (e.g. 'accession' or 'description')
    possible_group_fields = [c for c in main_df.columns if c.lower() in ['accession', 'description', 'label', 'sample']]

    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No group field found for grouping.")
else:
    print('No numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If a group_field exists, boxplot by group
    if 'group_field' in locals() and group_field:
        # Only plot if not too many groups
        n_groups = filtered_df[group_field].nunique()
        if n_groups <= 20:
            plt.figure(figsize=(max(6, n_groups), 4))
            sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
            plt.xticks(rotation=45)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded and explored the FAIR^2 mass spectrometry dataset of extracellular vesicles from human mast cells using the `mlcroissant` library. The dataset includes measurements such as molecular weight, abundance values, and protein identifications. We demonstrated filtering, normalization, grouping, and visualization with a focus on fields referenced by their `@id`s. For further analysis, explore domain-specific questions related to protein expression, modification, or bioinformatics hypotheses using the provided data structure.